# Feature Engineering para data/train.csv

Este notebook aplica transformaciones supervisadas y automáticas según el EDA realizado para data/train.csv.

Todos los pasos se adaptan dinámicamente a la estructura real del dataset.

## 1. Carga y Split de Datos

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Cargar datos
path = 'data/train.csv'
df = pd.read_csv(path)

# Eliminar columnas identificadoras si existen
def drop_idx_cols(df_):
    idx = [c for c in df_.columns if c.lower() in {'id', 'index'}]
    return df_.drop(idx, axis=1) if idx else df_

df = drop_idx_cols(df)

# Split (80/20, random_state=0)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=0)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

y_col = 'SalePrice'  # target

---

## 2. Imputación de Valores Faltantes

### 2.1. Categóricas con Mucho Missing

Variables categóricas con >5% de missing serán imputadas con 'Missing'. Las de <=5% con la moda de train.

In [ ]:
cat_cols = train_df.select_dtypes(include='object').columns.tolist() + [c for c in ['MSSubClass'] if c in train_df]
num_cols = [c for c in train_df.select_dtypes(include=[np.number]).columns if c != y_col and c not in ['MSSubClass']]

cat_na = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtFinType2',
    'BsmtExposure', 'BsmtFinType1', 'BsmtCond', 'BsmtQual', 'Electrical']

na_map = train_df[cat_cols].isna().mean()
cat_high_na = [c for c in cat_cols if na_map.get(c,0) > 0.05]
cat_low_na = [c for c in cat_cols if 0 < na_map.get(c,0) <= 0.05]

def impute_categoricals(train, test):
    for col in cat_high_na:
        val = 'Missing'
        train[col] = train[col].fillna(val)
        test[col] = test[col].fillna(val)
    for col in cat_low_na:
        moda = train[col].mode()[0]
        train[col] = train[col].fillna(moda)
        test[col] = test[col].fillna(moda)
    return train, test

train_df, test_df = impute_categoricals(train_df, test_df)

### 2.2. Numéricas con Missing

Variables numéricas con missing se imputan con la media de train y se crea un indicador de missing.

In [ ]:
num_na = ['LotFrontage', 'GarageYrBlt', 'MasVnrArea']

def impute_numerics(train, test):
    for col in num_na:
        if col not in train.columns: continue
        train[col+'_miss'] = train[col].isna().astype(int)
        test[col+'_miss'] = test[col].isna().astype(int)
        mean = train[col].mean()
        train[col] = train[col].fillna(mean)
        test[col] = test[col].fillna(mean)
    return train, test

train_df, test_df = impute_numerics(train_df, test_df)

---

## 3. Transformación de Variables Temporales (Años)

Calculamos "años transcurridos desde la venta" para:
- 'YearBuilt', 'YearRemodAdd', 'GarageYrBlt': usamos 'YrSold' como referencia.

In [ ]:
time_vars = ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt']
if 'YrSold' in train_df:
    ref_year_col = 'YrSold'
    for col in time_vars:
        if col in train_df:
            train_df[col+'_age'] = train_df[ref_year_col] - train_df[col]
            test_df[col+'_age'] = test_df[ref_year_col] - test_df[col]

---

## 4. Transformaciones sobre Variables Numéricas

### 4.1. Binarización para Dominadas por Ceros (>50%)

In [ ]:
zero_dom_cols = ['MasVnrArea', 'BsmtFinSF2', '2ndFlrSF', 'LowQualFinSF', 'BsmtFullBath', 'BsmtHalfBath',
                'HalfBath', 'WoodDeckSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal']
for c in zero_dom_cols:
    if c in train_df:
        train_df[c+'_bin'] = (train_df[c] > 0).astype(int)
        test_df[c+'_bin'] = (test_df[c] > 0).astype(int)

### 4.2. Transformación para Sesgo (Log1p o Yeo-Johnson)

Aplicamos Yeo-Johnson para variables sesgadas con ceros en los datos.

In [ ]:
from sklearn.preprocessing import PowerTransformer
skewed_cols = [
    'MiscVal', 'PoolArea', 'LotArea', '3SsnPorch', 'LowQualFinSF', 'KitchenAbvGr',
    'BsmtFinSF2', 'ScreenPorch', 'BsmtHalfBath', 'EnclosedPorch', 'MasVnrArea',
    'OpenPorchSF', 'LotFrontage', 'BsmtFinSF1', 'WoodDeckSF', 'TotalBsmtSF',
    'MSSubClass', '1stFlrSF', 'GrLivArea', 'BsmtUnfSF', '2ndFlrSF'
]
# Limitar a variables presentes
skewed_cols = [c for c in skewed_cols if c in train_df]

pt = PowerTransformer(method='yeo-johnson')
for col in skewed_cols:
    tr_data = train_df[[col]].values
    te_data = test_df[[col]].values
    pt.fit(tr_data)
    train_df[col+'_yj'] = pt.transform(tr_data)
    test_df[col+'_yj'] = pt.transform(te_data)

---

## 5. Encoding de Variables Categóricas

### 5.1. Ordinal Encoding basado en sentido de calidad (según EDA y valores del dominio)

In [ ]:
ordinal_map = {
    'ExterQual': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'ExterCond': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'BsmtQual': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'BsmtCond': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'HeatingQC': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'KitchenQual': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'FireplaceQu': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'GarageQual': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'GarageCond': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
}
for col, mapping in ordinal_map.items():
    if col in train_df:
        train_df[col+'_OE'] = train_df[col].map(mapping)
        test_df[col+'_OE'] = test_df[col].map(mapping)

### 5.2. Rare Label Encoder para nominales de alta cardinalidad (>10)

In [ ]:
# Detectarlas según cardinalidad real
card_counts = train_df[cat_cols].nunique()
high_card = card_counts[card_counts > 10].index.tolist()
def rare_label_encoder(train, test, cols, threshold=0.01):
    # Etiquetar como Rare
    for col in cols:
        freq = train[col].value_counts(normalize=True)
        frequent = freq[freq >= threshold].index
        train[col+'_rl'] = np.where(train[col].isin(frequent), train[col], 'Rare')
        test[col+'_rl'] = np.where(test[col].isin(frequent), test[col], 'Rare')
    return train, test

train_df, test_df = rare_label_encoder(train_df, test_df, high_card)

### 5.3. Ordinal Encoder global para todas las categóricas (nominales y la resultante de rare label)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
# Incluir rarelabelizadas y categóricas simples no ordinales
encoded_cols = []
# Rarelabel encoded:
encoded_cols += [c+'_rl' for c in high_card if c+ '_rl' in train_df]
# Simple nominales (no ordinales):
simple_nominals = [c for c in cat_cols if c not in ordinal_map]
encoded_cols += [c for c in simple_nominals if all(c+x not in train_df for x in ['_rl','_OE'])]

# Fitear ordinalencoder solo en train y aplicarlo a test sin leakage
OE = OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)
OE.fit(train_df[encoded_cols])
train_df[encoded_cols] = OE.transform(train_df[encoded_cols])
test_df[encoded_cols] = OE.transform(test_df[encoded_cols])

---

## 6. Escalado de Todas las Variables (MinMax)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

features = [c for c in train_df.columns if c != y_col]
scaler = MinMaxScaler()
scaler.fit(train_df[features])
train_df[features] = scaler.transform(train_df[features])
test_df[features] = scaler.transform(test_df[features])

---

## 7. Validación Final: No NaN, No Inf

In [ ]:
import warnings
for name, df_check in [('train', train_df), ('test', test_df)]:
    if df_check.isna().sum().sum()>0 or np.isinf(df_check.to_numpy()).sum()>0:
        warnings.warn(f'¡{name} set contiene NaN o infinitos!')
    else:
        print(name, 'OK: sin NaN ni Inf.')

---

## 8. Guardar Procesados (opcional)

In [ ]:
# Descomentar si se quieren guardar
# train_df.to_csv('data/train_fe.csv', index=False)
# test_df.to_csv('data/test_fe.csv', index=False)